In [ ]:
# ============================================================
# 🥷 AIBO v7.4.0 ワンプッシュ起動セル
# Phase 1: HF cache · Phase 2: 依存 · Phase 3: 起動 · Phase 4: 完成 · Phase 5: API 公開
# ============================================================
import os, sys, importlib, subprocess, shutil, time

# Phase 0: import warm-up は無効化 (VRAM OOM の原因となるため)
# torch/nunchaku を background thread で import すると CUDA コンテキスト初期化が
# 競合し、main thread のメモリプール確保が失敗するケースを確認。
# 4 秒の最適化を捨てて VRAM 安定性を優先。

# ─── Phase 1: Drive mount + HF cache ─────────────────────────
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=False)
# 🆕 CASE-A (RECON-002 第2段 · 案A): モデルは HF Hub から /content(NVMe)へ直 DL する。
# Drive FUSE 越しの巨大 safetensors 読み込み(=truncation→外国人化)を構造的に発生させない。
# Drive はコード(AIBO_ROOT)の置き場としてのみ使い、モデルの一次ソースからは外す。
GDRIVE_HF_CACHE = "/content/drive/MyDrive/aibo_hf_cache"   # sync_back_to_drive() 用に定義のみ保持
CONTENT_HF_CACHE = "/content/aibo_hf_cache"
# 🛡️ FIX-1 (IMPL-003 / H1 対策): /content/aibo_hf_cache が「Drive を指す stale symlink」だと
#   直 DL が Drive に書かれ、ロードで FUSE mmap → OSError errno 19(ENODEV)で落ちる。
#   symlink なら必ず除去してから実ディレクトリを作り、Drive 非経由をアサートする(silent fail 禁止)。
if os.path.islink(CONTENT_HF_CACHE):
    os.unlink(CONTENT_HF_CACHE)          # 旧 notebook が作った Drive 直結 symlink を除去
elif os.path.isdir(CONTENT_HF_CACHE) and os.path.realpath(CONTENT_HF_CACHE).startswith("/content/drive"):
    raise RuntimeError(f"[CASE-A] HF cache が Drive 配下: {os.path.realpath(CONTENT_HF_CACHE)}")
os.makedirs(CONTENT_HF_CACHE, exist_ok=True)
# ★ ガード: 用意した cache が Drive を指していないことを起動時にアサート
_real_cache = os.path.realpath(CONTENT_HF_CACHE)
if _real_cache.startswith("/content/drive"):
    raise RuntimeError(f"[CASE-A] FATAL: HF cache が Drive 上 ({_real_cache})。NVMe 直 DL に失敗。")
print(f"✅ [CASE-A] HF cache = {_real_cache} (NVMe・Drive 非経由を確認)")

# 🗑️ CASE-A で撤去した事故経路:
#   - tar pipe(CACHE_SYNC_TIMEOUT=180)による Drive→/content 一括コピー
#   - timeout 時の shutil.rmtree(CONTENT_HF_CACHE)+os.symlink(GDRIVE→CONTENT)(=Drive 直結)
#   これらが「180s で部分コピー→直結 symlink→FUSE 越し巨大読み込み→truncation」の確定的事故経路だった。

# /root/.cache/huggingface → /content/aibo_hf_cache symlink
# (env 変数を無視する library 対策、全 HF DL を NVMe に固定。Drive ではなく NVMe を指す)
ROOT_HF = "/root/.cache/huggingface"
os.makedirs("/root/.cache", exist_ok=True)
if os.path.islink(ROOT_HF) or os.path.exists(ROOT_HF):
    subprocess.run(["rm", "-rf", ROOT_HF], check=False)
os.symlink(CONTENT_HF_CACHE, ROOT_HF)
print(f"✅ HF cache symlink: {ROOT_HF} → {CONTENT_HF_CACHE}")

# env 変数も /content (NVMe) を指す
for k in ["HF_HOME", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE", "HF_HUB_CACHE"]:
    os.environ[k] = CONTENT_HF_CACHE
# 🆕 CASE-A: HF_HUB_ENABLE_HF_TRANSFER は deprecated 警告が出るため設定しない。
#   高速 DL は hf_xet(既定・huggingface_hub>=0.32)に委ねる。
#   hf_xet の稀なサイズ一致破損は C0 検証ゲート(IMPL-001)が from_pretrained 直前に捕捉する前提。
print(f"✅ HF cache → {CONTENT_HF_CACHE} (NVMe)")

# ─── Phase 1.4: HF token 注入 (userdata 経由・コードに直書きしない) ───
# gated 2 repo(FLUX.1-dev / FLUX.1-Redux-dev)の DL に HF token が必須(G0 調査)。
try:
    from google.colab import userdata
    _hf_tok = userdata.get('HF_TOKEN')   # 既存の NGROK_AUTH_TOKEN と同じ Colab Secrets 仕組み
except Exception as _e:
    _hf_tok = None
    print(f"⚠️ [CASE-A] userdata.get('HF_TOKEN') 取得失敗: {type(_e).__name__}: {_e}")
if _hf_tok:
    os.environ['HF_TOKEN'] = _hf_tok        # 値はログに出さない
    try:
        from huggingface_hub import login
        login(token=_hf_tok)
        print("✅ [CASE-A] HF login OK (token は非表示)")
    except Exception as _e:
        print(f"⚠️ [CASE-A] HF login 失敗: {type(_e).__name__}: {_e}")
else:
    # 握りつぶさない: token 不在を明示警告(gated DL は 401/403 で失敗する)
    print("⚠️ [CASE-A] HF_TOKEN 未設定。gated モデル(FLUX.1-dev/Redux)の DL は失敗します")

# ─── Phase 1.5: HF Hub → /content 直 DL (snapshot_download 一次化) ───
# default cache-dir 方式(local_dir 指定しない=巨大単一ファイルの resume 堅牢性のため · RECON-002 F3)。
# 出口は C0 検証ゲート(IMPL-001)が各 from_pretrained 直前に守る。
from huggingface_hub import snapshot_download
# (repo_id, allow_patterns, ignore_patterns) — None は無指定(allow=None で repo 丸ごと / ignore=None で除外なし)。
_HF_REPOS = [
    # gated (HF_TOKEN 必須)
    ("black-forest-labs/FLUX.1-dev", None, ["flux1-dev.safetensors", "transformer/*"]),  # IMPL-009: bf16 transformer 2塊を除外(Nunchaku INT4 使用)
    ("black-forest-labs/FLUX.1-Redux-dev", None, None),
    # public (token 不要)
    # 🛡️ FIX-2 (IMPL-003 / H2 対策): nunchaku は int4 をピン。A100(Ampere)= int4(get_precision 準拠)。
    #   fp4 は Blackwell 専用なので落とさない。無駄 DL(fp4 ~6.5GB)削減 + DL/ロードの precision 整合。
    ("nunchaku-tech/nunchaku-flux.1-dev",          # INT4 transformer 本体(307→nunchaku-ai に自動追従)
     ["svdq-int4_r32-flux.1-dev.safetensors", "*.json", "README*"], None),
    ("mit-han-lab/svdq-int4-flux.1-fill-dev", None, None),       # Fill transformer
    ("Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0", None, None),
    ("XLabs-AI/flux-ip-adapter", None, None),
    ("guozinan/PuLID", None, None),
    ("ByteDance/Hyper-SD", ["Hyper-FLUX.1-dev-8steps-lora.safetensors"], None),  # IMPL-009: 8steps LoRA 単一化
]
print(f"\n📥 [CASE-A] HF Hub → /content 直 DL 開始 ({len(_HF_REPOS)} repo)")
_t_dl = time.time()
for _repo, _allow, _ignore in _HF_REPOS:
    try:
        snapshot_download(repo_id=_repo, allow_patterns=_allow, ignore_patterns=_ignore)   # allow/ignore=None は無指定。既ロード分は resume/skip。実体は /content(NVMe)
        print(f"  ✅ [CASE-A] OK: {_repo}")
    except Exception as _e:
        # 握りつぶさない: gated は token 無/未同意で 401/403。明示 fail-fast(縮退に逆戻りさせない)。
        print(f"  ❌ [CASE-A] FAIL: {_repo} :: {type(_e).__name__}: {_e}")
        raise
print(f"✅ [CASE-A] 直 DL 完了: {time.time()-_t_dl:.1f}s")

# ─── Phase 2: torchsde 事前 install ─────────────────────────
for dep in ['torchsde']:
    try:
        __import__(dep)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", dep], check=True)

# ─── Phase 3: AIBO 起動 ─────────────────────────────────────
print("\n" + "=" * 60)
print("🚀 AIBO v7.4.0 起動シーケンス")
print("=" * 60)

# ─── code transport = GitHub(exact commit・DriveFS clobber 根絶 / stale-clone 根治)───
#   code は Drive(DriveFS)から取得しない（古い local キャッシュを cloud に押し戻す clobber の温床）。
#   ★毎回 fresh clone(確実削除 → 空パスへ clone → reset --hard/clean で working tree を blob から強制復元)。
#     旧症状(2026-06-23): rm -rf 不全 → 残骸の上に重ねて clone → HEAD/blob は正なのに working file だけ旧版。
#   data(refs/outputs)=Drive、models=HF は不変。
import base64, hashlib, re
import json as _json
print("🧭 [Cell0] code-transport = GitHub FRESH-CLONE 版 (rev 2026-06-23 stale-root-fix) ─ この行が無ければ旧 notebook を開いている", flush=True)
from google.colab import userdata as _udata
_PAT = _udata.get('GH_PAT')                       # Colab Secrets。値はログにも remote にも出さない
AIBO_PIN = "sync/colab"                            # 既定=branch tip。特定版に固定したい時は commit SHA を入れる
_SRC, _BR = "/content/aibo_src", "sync/colab"
_CLEAN_URL = "https://github.com/miya390831-a11y/aibo_v8.git"   # PAT 非埋め込み(remote/ログに残す URL)

# ── 2-1: 確実な削除(残骸の上に重ねない)── shutil + rm -rf の二重削除 → 残存なら STOP ──
print(f"🧹 [transport] 確実削除 → 空パスへ fresh clone: {_SRC}", flush=True)
shutil.rmtree(_SRC, ignore_errors=True)
subprocess.run(["rm", "-rf", _SRC], check=False)
if os.path.exists(_SRC):   # 削除しきれない=別プロセスが掴んでいる等。原因を表面化(silent 禁止)
    raise RuntimeError(f"[transport] FATAL: 削除後も {_SRC} が残存。掴んでいるプロセスがある可能性 → ランタイム再起動が必要")

# ── 2-4: PAT を URL/remote に埋めない。http.extraHeader(Basic)で渡し、remote は PAT 非埋め込み URL に固定 ──
if not _PAT:
    raise RuntimeError("[transport] FATAL: GH_PAT 未設定(Colab Secrets)。clone できません")
_auth = base64.b64encode(f"x-access-token:{_PAT}".encode()).decode()   # 値はログに出さない
_hdr = f"http.extraHeader=AUTHORIZATION: Basic {_auth}"
# 注意: argv に _hdr(base64 PAT)が載るため、この subprocess の argv はログに出さない(漏洩防止)。
subprocess.run(["git", "-c", _hdr, "clone", "--quiet", "--branch", _BR, _CLEAN_URL, _SRC], check=True)
subprocess.run(["git", "-C", _SRC, "remote", "set-url", "origin", _CLEAN_URL], check=True)  # remote=PAT 非埋め込み
if AIBO_PIN != _BR:        # SHA 指定時のみ固定(fresh clone は既に branch tip)
    subprocess.run(["git", "-c", _hdr, "-C", _SRC, "fetch", "--quiet", "origin", AIBO_PIN], check=False)
    subprocess.run(["git", "-C", _SRC, "checkout", "--quiet", "--force", AIBO_PIN], check=True)
del _PAT, _auth, _hdr      # メモリからも PAT 派生物を落とす

# ── 2-2: working tree を HEAD(=正 blob)から強制復元。index は正・ディスクだけ旧、を構造的に是正 ──
subprocess.run(["git", "-C", _SRC, "reset", "--hard", "HEAD"], check=True)
subprocess.run(["git", "-C", _SRC, "clean", "-fdx"], check=True)
_sha = subprocess.run(["git", "-C", _SRC, "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
AIBO_ROOT = _SRC           # repo 直下に *.py(repo root = code)
print(f"✅ [code=GitHub] {_BR} @ {_sha[:8]} -> {AIBO_ROOT}（fresh clone・reset --hard 済・DriveFS 非経由）")

# ── 2-3: clone 直後 md5 assert(silent fail 禁止)── ディスク実体 == committed manifest を Cell 0 自身が証明 ──
def _norm_md5(path):
    """改行正規化(CRLF/CR→LF)md5。tools/check_manifest._norm_md5 と同一規則。"""
    with open(path, "rb") as _f:
        _raw = _f.read()
    _text = _raw.decode("utf-8", errors="replace").replace("\r\n", "\n").replace("\r", "\n")
    return hashlib.md5(_text.encode("utf-8")).hexdigest()
_man_path = os.path.join(AIBO_ROOT, "module_manifest.json")
with open(_man_path, "r", encoding="utf-8") as _f:
    _man = _json.load(_f).get("modules", {})
_mismatch = []
for _fn, _exp in sorted(_man.items()):
    _fp = os.path.join(AIBO_ROOT, _fn)
    if not os.path.exists(_fp):
        _mismatch.append(f"{_fn}: MISSING (expected {_exp[:8]})"); continue
    _act = _norm_md5(_fp)
    if _act != _exp:
        _mismatch.append(f"{_fn}: disk={_act[:8]} != manifest={_exp[:8]}")
if _mismatch:
    raise RuntimeError("[transport] FATAL: clone 直後の disk md5 != committed manifest(stale clone)。"
                       " build に進ませない:\n  " + "\n  ".join(_mismatch))
print(f"✅ [transport] disk md5 == committed manifest（{len(_man)} module・stale でないことを証明）", flush=True)

# BOM 除去
for fname in ["01_config.py", "02_colab_setup.py", "03_identity_engine.py",
              "04_pipeline_manager.py", "05_orchestrator.py", "06_ui.py",
              "07_main.py", "08_face_refiner.py"]:
    fpath = os.path.join(AIBO_ROOT, fname)
    with open(fpath, "rb") as f:
        c = f.read()
    if c.startswith(b'\xef\xbb\xbf'):
        with open(fpath, "wb") as f:
            f.write(c[3:])

# ── sys.path 衛生: AIBO_ROOT を最優先(index0)に固定し、AIBO コアを含む旧 code path を除去 ──
#   旧 Drive code path が sys.path 上位に残ると、purge 後の再 import が旧版を解決してしまう。
#   旧実装 `if AIBO_ROOT not in sys.path: insert(0)` は、AIBO_ROOT が既に下位に在ると先頭へ
#   入れ直さず、上位の stale path が shadow した(第2症状の一因)。常に [AIBO_ROOT] を先頭に固定。
_CORE_BASENAMES = {
    "01_config.py", "02_colab_setup.py", "03_identity_engine.py", "04_pipeline_manager.py",
    "05_orchestrator.py", "06_ui.py", "07_main.py", "08_face_refiner.py", "09_fastapi_server.py",
    "10_mask_engine.py", "11_pulid_cn_pipeline.py", "15_makeup_engine.py", "16_pose_extractor.py",
    "17_neutralize_engine.py",
}
_aibo_real = os.path.realpath(AIBO_ROOT)
_clean_path = []
for _p in sys.path:
    try:
        _rp = os.path.realpath(_p) if _p else _p
    except Exception:
        _clean_path.append(_p); continue
    if _rp == _aibo_real:
        continue   # 後で先頭に入れ直す(重複・順序ずれを排除)
    if _p and os.path.isdir(_p) and any(os.path.exists(os.path.join(_p, _b)) for _b in _CORE_BASENAMES):
        print(f"🧹 [transport] stale code path を sys.path から除外: {_p}", flush=True)
        continue
    _clean_path.append(_p)
sys.path[:] = [AIBO_ROOT] + _clean_path

# ── 2-5(強化): メモリ stale 対策。名前 regex(^\d{2}_)では捕まらない別名/別パス(Drive 旧 code)
#   経由のモジュールも __file__/basename で根こそぎ purge。旧実装は取り逃しが残り、import 0.07s で
#   旧版を掴んだ(2026-06-23 第2症状: disk は正なのに runtime md5=3b297f80)。GPT 診断 purge_python_stale 準拠。
def _is_aibo_module(_name, _mod):
    if re.match(r"^\d{2}_", _name):                       # 名前一致(従来)
        return True
    _f = getattr(_mod, "__file__", None)
    if not _f:
        return False
    _rf = os.path.realpath(_f)
    return _rf.startswith(_aibo_real) or os.path.basename(_rf) in _CORE_BASENAMES  # __file__ 配下 or コア basename
def purge_python_stale():
    """__file__ ベースで AIBO モジュールを sys.modules から全消し(別名/別パス経路も捕捉)。"""
    _k = []
    for _name, _mod in list(sys.modules.items()):
        if _is_aibo_module(_name, _mod):
            del sys.modules[_name]
            _k.append((_name, getattr(_mod, "__file__", None)))
    importlib.invalidate_caches()
    return _k
_killed = purge_python_stale()
print(f"🧹 [transport] sys.modules purge: {len(_killed)} module 削除", flush=True)
for _n, _f in _killed:
    print(f"    - {_n}  <-  {_f}", flush=True)

# ── 強制再起動ゲート(import 前): purge 後もメモリが汚れていたら in-process 修復不能 → ランタイム再起動 ──
#   依頼(2): 「それでも残るなら os.kill 的な強制再起動」「import 前にメモリが汚れてたら STOP」を実装。
_still = [(_n, getattr(_m, "__file__", None)) for _n, _m in list(sys.modules.items())
          if _is_aibo_module(_n, _m)]
if _still:
    print("❌ [transport] FATAL: purge 後も AIBO モジュールが sys.modules に残存(in-process 修復不能):", flush=True)
    for _n, _f in _still:
        print(f"    - {_n}  <-  {_f}", flush=True)
    print("🔄 [transport] ランタイムを強制再起動します(os.kill -9)。再起動後にこのセルを再実行してください。", flush=True)
    sys.stdout.flush()
    os.kill(os.getpid(), 9)   # Colab はランタイム再起動 → 次回 Cell 0 はクリーンメモリで import(~20s)
t_imp = time.perf_counter()
mod01 = importlib.import_module("01_config")
mod02 = importlib.import_module("02_colab_setup")  # numpy 自動チェック + 必要時のみ自動再起動
mod03 = importlib.import_module("03_identity_engine")
mod04 = importlib.import_module("04_pipeline_manager")
mod05 = importlib.import_module("05_orchestrator")
mod06 = importlib.import_module("06_ui")
mod07 = importlib.import_module("07_main")
mod08 = importlib.import_module("08_face_refiner")
_imp_s = time.perf_counter() - t_imp
print(f"⏱️ import: {_imp_s:.2f}s")

# ── runtime 検証ゲート(import 後): ロード済みコアの __file__ と md5 を committed manifest と照合 ──
#   ★これが真の stale 判定(依頼3)。disk が正でも、ロードされた module object が旧パス/旧版なら STOP。
#   import 0.07s は本物(~20s)でない兆候。ただし warm runtime では正規 import も速いため、時間単独では
#   STOP しない(誤再起動を避ける)。md5/__file__ 一致を確定条件にし、不一致なら強制再起動。
_loaded_bad = []
for _mod in (mod01, mod02, mod03, mod04, mod05, mod06, mod07, mod08):
    _name = _mod.__name__
    _f = getattr(_mod, "__file__", None)
    _rf = os.path.realpath(_f) if _f else None
    if not _rf or not _rf.startswith(_aibo_real):
        _loaded_bad.append(f"{_name}: __file__={_f}(AIBO_ROOT 外=旧パス)"); continue
    _exp = _man.get(_name + ".py")
    if _exp is None:
        continue
    _act = _norm_md5(_rf)
    if _act != _exp:
        _loaded_bad.append(f"{_name}: runtime md5={_act[:8]} != manifest={_exp[:8]}  __file__={_f}")
if _loaded_bad:
    print("❌ [transport] FATAL: runtime モジュール != committed manifest(stale import 確定):", flush=True)
    for _b in _loaded_bad:
        print(f"    - {_b}", flush=True)
    print(f"   import={_imp_s:.2f}s(本物は ~20s)・メモリ stale。", flush=True)
    print("🔄 [transport] ランタイムを強制再起動します(os.kill -9)。再起動後にこのセルを再実行してください。", flush=True)
    sys.stdout.flush()
    os.kill(os.getpid(), 9)
print(f"✅ [transport] runtime md5 == committed manifest(全 8 コア・stale import なし)", flush=True)
if _imp_s < 5.0:
    print(f"ℹ️ [transport] import 高速({_imp_s:.2f}s)だが runtime md5 検証済 → stale ではない(warm deps)。", flush=True)

SystemConfig = mod01.SystemConfig
GenerationConfig = mod01.GenerationConfig
IdentityConfig = mod01.IdentityConfig
StudioMode = mod01.StudioMode
AiboMain = mod07.AiboMain

# AiboMain 起動
t_boot = time.perf_counter()
aibo = AiboMain()
if hasattr(aibo, "run"):
    aibo.run(enable_gradio=False)  # A方式運用 · Phase G スキップ
print(f"⏱️ AiboMain 起動: {time.perf_counter() - t_boot:.2f}s")

orchestrator = aibo.orchestrator
pm = orchestrator.pm
ie = orchestrator.ie
ipa = ie.ip_adapter
gen_cfg = GenerationConfig()
id_cfg = IdentityConfig()

# ─── Phase 4: Phase 1 完成状態セットアップ ──────────────────
print("\n" + "=" * 60)
print("🔧 Phase 1 完成状態セットアップ")
print("=" * 60)
tf = pm._shared_transformer
if pm.pipe_cnet is None:
    pm.ensure_controlnet()
pm._cn_forward_wrapped = False
pm._wrap_transformer_forward_for_cn()
if pm.pipe_cnet.image_encoder is None:
    pm.pipe_cnet.image_encoder = ipa._image_encoder
    pm.pipe_cnet.feature_extractor = ipa._feature_extractor
ipa.set_scale(pm.pipe_base, id_cfg.ip_adapter_weight)
ipa.set_scale(pm.pipe_cnet, id_cfg.ip_adapter_weight)
print(f"✅ forward={tf.forward.__qualname__}")
print(f"✅ IP-Adapter scale={id_cfg.ip_adapter_weight}")

print("\n" + "=" * 60)
print("🎉 v7.4.0 Phase 1 完成 · 即生成可能!")
print("=" * 60)
import sys as _sys
_sys.stdout.flush()

# ─── Phase 5: FastAPI + ngrok 公開 ───────────────────────────
import traceback as _tb
_sys.stdout.flush()
print("\n" + "=" * 60, flush=True)
print("🚀 FastAPI + ngrok 公開", flush=True)
print("=" * 60, flush=True)

try:
    import threading, requests
    for pkg in ["fastapi", "pyngrok"]:
        try:
            __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

    from pyngrok import ngrok, conf
    from google.colab import userdata

    srv = importlib.import_module("09_fastapi_server")

    # 既起動チェック
    already_running = False
    try:
        r = requests.get("http://localhost:8000/api/system/status", timeout=2)
        if r.status_code == 200 and r.json().get("orchestrator_attached"):
            already_running = True
            print("ℹ️ FastAPI 既起動、スキップ", flush=True)
    except: pass

    if not already_running:
        srv.attach_orchestrator(orchestrator, pm)
        threading.Thread(
            target=lambda: srv.run_server(host="0.0.0.0", port=8000, log_level="warning"),
            daemon=True, name="aibo-fastapi"
        ).start()
        time.sleep(3)

    r = requests.get("http://localhost:8000/api/system/status", timeout=5)
    b = r.json()
    print(f"✅ FastAPI · GPU={b['gpu_name']} · VRAM={b['vram_used_gb']:.1f}/{b['vram_total_gb']:.1f} GB", flush=True)

    # ngrok
    conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN')
    try:
        for t in ngrok.get_tunnels():
            ngrok.disconnect(t.public_url)
        ngrok.kill()
        time.sleep(1)
    except: pass

    public_url = ngrok.connect(8000, "http").public_url
    time.sleep(3)

    # ngrok 疎通確認 (リトライ付き)
    for i in range(5):
        try:
            r = requests.get(f"{public_url}/api/system/status",
                             headers={"ngrok-skip-browser-warning": "true"}, timeout=10)
            if r.status_code == 200:
                break
        except: pass
        time.sleep(2)

    env_content = f"NEXT_PUBLIC_API_URL={public_url}\nNEXT_PUBLIC_NGROK_SKIP_WARNING=true"
    with open(f"{AIBO_ROOT}/.env.local.latest.txt", "w") as f:
        f.write(env_content)

    print(f"\n🔌 API URL: {public_url}", flush=True)
    print("\n" + "=" * 60, flush=True)
    print("📋 PC の .env.local にコピー", flush=True)
    print("=" * 60, flush=True)
    print(env_content, flush=True)
    print("\n" + "=" * 60, flush=True)
    print("🌐 ブラウザで開く (PC で npm run dev が走ってる前提)", flush=True)
    print("=" * 60, flush=True)
    print("http://localhost:3000", flush=True)
    print("\n🎉 ワンプッシュ起動完了 🥷", flush=True)
except Exception as _e:
    print(f"\n❌ Phase 5 で例外発生: {type(_e).__name__}: {_e}", flush=True)
    _tb.print_exc()
    raise

def sync_back_to_drive():
    """/content cache の内容を Drive に書き戻し (新規 DL の永続化用)"""
    if os.path.islink(CONTENT_HF_CACHE):
        print("ℹ️ symlink モード · Drive 直結のため sync 不要")
        return
    t = time.time()
    try:
        _cmd = f'tar cf - -C "{CONTENT_HF_CACHE}" . | tar xf - -C "{GDRIVE_HF_CACHE}"'
        subprocess.run(["bash", "-c", _cmd], capture_output=True, text=True, timeout=300)
        print(f"✅ Drive 同期完了 (tar): {time.time()-t:.1f}s")
    except subprocess.TimeoutExpired:
        print(f"⚠️ Drive 同期 timeout (300s) · 手動で再試行してください")
    except Exception as _e:
        print(f"⚠️ Drive 同期失敗: {_e}")


In [ ]:
# ============================================================
# 🔍 環境確認 (Cell 0 実行後 · 任意)
# ============================================================
import json
import urllib.request
from pathlib import Path

AIBO_ROOT = "/content/aibo_src"  # code=GitHub clone(DriveFS 非経由)
print("=" * 60)
print("🔍 環境確認")
print("=" * 60)

# 1. FastAPI (local)
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/api/system/status", timeout=10) as resp:
        st = json.loads(resp.read().decode())
    print("✅ localhost:8000")
    print(f"   orchestrator_attached: {st.get('orchestrator_attached')}")
    if st.get("gpu_available"):
        print(f"   GPU: {st.get('gpu_name')}")
        print(f"   VRAM: {st.get('vram_used_gb')}/{st.get('vram_total_gb')} GB ({st.get('vram_pct')}%)")
except Exception as exc:
    print(f"❌ localhost:8000 · {exc}")

# 2. .env.local.latest.txt + ngrok 疎通
env_file = Path(AIBO_ROOT) / ".env.local.latest.txt"
if env_file.is_file():
    text = env_file.read_text(encoding="utf-8").strip()
    print(f"\n✅ {env_file.name}:")
    print(text)
    ngrok_url = None
    for line in text.splitlines():
        if line.startswith("NEXT_PUBLIC_API_URL="):
            ngrok_url = line.split("=", 1)[1].strip()
            break
    if ngrok_url:
        try:
            req = urllib.request.Request(
                f"{ngrok_url.rstrip('/')}/api/system/status",
                headers={"ngrok-skip-browser-warning": "true"},
            )
            with urllib.request.urlopen(req, timeout=15) as resp:
                ext = json.loads(resp.read().decode())
            print(f"\n✅ ngrok 疎通 OK · orchestrator_attached={ext.get('orchestrator_attached')}")
        except Exception as exc:
            print(f"\n⚠️ ngrok 疎通: {exc}")
else:
    print(f"\n⚠️ {env_file} がありません (Cell 0 を先に実行)")
